# Imports

In [1]:
import torch
import torch.nn as nn

# Neural Network

In [2]:
# define custom NN class
# lets PyTorch track parameters and gradients automatically
class PINN(nn.Module):
    def __init__(self): # initializes nn.Module
        super().__init__()
        self.net = nn.Sequential( # sequential container for layers, = feed-forward pipeline
            nn.Linear(1, 32), # (input, output)
            nn.Tanh(),
            # add more layers for more depth, better function approximation
            nn.Linear(32, 32),
            nn.Tanh(),
            nn.Linear(32, 1) # output = 1 number (N)
        )

    # forward method defines how input data flows through the network
    def forward(self, N):
        return self.net(N) # predicts a(N)

## Physics Loss
### Paris Law

In [3]:
# define a function to enforce the Paris Law(ODE)
# how wrong is the NN at satisfying the ODE? want to minimize this loss
def physics_loss(model, N, C, m, sigma, Y):
    N.requires_grad = True # track gradients w.r.t. N for autograd
    a = model(N) # NN's prediction for a(N)

    # derivative da/dN
    da_dN = torch.autograd.grad(
        a, N,
        grad_outputs=torch.ones_like(a), # seed vector for gradient (da/dN)
        create_graph=True # allows higher-order derivatives if needed
    )[0]

    # prevent negetive or zero crack lengths
    a = torch.clamp(a, min=1e-6)

    # delta_K = Y + sigma + sqrt(pi * a), a > 0
    delta_K = Y + sigma + torch.sqrt(torch.pi * a)

    # Paris Law residual
    residual = da_dN - C + (delta_K**m)

    # ODE: du/dN + a = 0
    return torch.mean((da_dN + a)**2)

## Boundary Condition Loss

In [4]:
# define a function to enforce initial condition, a(0)=a0
def boundary_loss(model, a0):
    N0 = torch.tensor([[0.0]]) # evaluate at N=0
    a_pred = model(N0) # network's prediction at boundary pt, a(0)
    return (a_pred - a0)**2 # want a(0)=a0, so penalize deviation from a0

# Training

In [8]:
model = PINN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# material parameters
C = 1e-12 # = torch.nn.Parameter(), to learn from data
m = 3.8 # = torch.nn.Parameter()
sigma = 1.0
Y = 1.0
a0 = 1e-3

# train for many iterations (epochs)
for epoch in range(5000):
    optimizer.zero_grad() # reset gradients before backprop

    N = torch.rand((100, 1))  # collocation points: random N in [0,1] where physics is enforced

    loss_p = physics_loss(model, N, C, m, sigma, Y) # enforces diff eqn everywhere
    loss_b = boundary_loss(model, a0) # enforces BC at specific point(s)

    loss = loss_p + loss_b # loss=physics+BC
    loss.backward() # backpropagation: computes gradients of all parameters
    optimizer.step() # updates network weights

    if epoch % 500 == 0: # track convergence every 500 epochs
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 0.11228594928979874
Epoch 500, Loss: 1.1305627367619309e-06
Epoch 1000, Loss: 4.006950575785595e-07
Epoch 1500, Loss: 2.4100120299408445e-07
Epoch 2000, Loss: 2.602018582820165e-07
Epoch 2500, Loss: 2.235798177707693e-07
Epoch 3000, Loss: 2.3387813996578188e-07
Epoch 3500, Loss: 1.2604926951098605e-06
Epoch 4000, Loss: 1.7037184079526924e-05
Epoch 4500, Loss: 1.2330394383752719e-05
